# Comparação com vs sem split temporal + expansão via API

Este notebook documenta e visualiza os resultados da comparação entre **três protocolos de avaliação** para os mesmos modelos treinados em [`05_modelos_lightgbm_arvore_rf.ipynb`](05_modelos_lightgbm_arvore_rf.ipynb):

1. **Com split temporal** — holdout por data (treino até 09/03, teste 10–11/03) + `TimeSeriesSplit` no treino
2. **Sem split temporal (80/20 cronológico)** — como em `lightgbm.ipynb`: últimos 20% das linhas
3. **Sem split temporal (80/20 aleatório)** — `train_test_split(shuffle=True)`, que **vaza informação temporal**

Também registra a tentativa de **expandir o dataset via API Polymarket** e as limitações encontradas.

Dados numéricos em `reports/split_comparison.json` e `reports/model_report_data.json`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
MODEL_ORDER = ["LightGBM", "Decision Tree", "Random Forest"]
MODEL_LABELS = {"LightGBM": "LightGBM", "Decision Tree": "Árvore de Decisão", "Random Forest": "Random Forest"}

PROTOCOLS = {
    "com_split_temporal": "Com split temporal",
    "sem_split_cronologico_8020": "80/20 cronológico",
    "sem_split_aleatorio": "80/20 aleatório",
}

## 1. Protocolos de avaliação

Usamos os **mesmos 3 modelos, 12 features e hiperparâmetros** do notebook 05.

| Protocolo | Descrição | Tamanho do teste |
|-----------|-----------|------------------|
| **Com split temporal** | Treino até 09/03, holdout 10–11/03 + TSCV no treino | 638k linhas (futuro real) |
| **Sem split (80/20 cronológico)** | Como `lightgbm.ipynb`: últimos 20% das linhas | 1,12M linhas (mistura de dias) |
| **Sem split (80/20 aleatório)** | `train_test_split(shuffle=True)` — vaza tempo | 1,12M linhas |

O target continua desbalanceado (~73% classe 0). Por isso priorizamos **ROC-AUC, F1, Precisão e Recall** em vez de acurácia isolada.

In [ ]:
def load_report(name: str) -> dict:
    path = Path("reports") / name
    if not path.exists():
        raise FileNotFoundError(f"Execute scripts/compare_split_protocols.py ou run_split_comparison_fast.py")
    return json.loads(path.read_text(encoding="utf-8"))


comparison = load_report("split_comparison.json")
model_report = load_report("model_report_data.json")

for key, label in PROTOCOLS.items():
    meta = comparison[key]["meta"]
    print(f"{label}: treino={meta['train_rows']:,} | teste={meta['test_rows']:,} | {meta['test_period']}")

## 2. Holdout — ROC-AUC por modelo

| Modelo | Com split temporal | 80/20 cronológico | Δ | 80/20 aleatório | Δ |
|--------|------------------:|------------------:|--:|----------------:|--:|
| LightGBM | 0,906 | 0,894 | −0,012 | 0,909 | +0,004 |
| Random Forest | 0,911 | 0,892 | −0,019 | 0,923 | +0,012 |
| Árvore de Decisão | 0,902 | 0,881 | −0,022 | 0,919 | +0,017 |

Δ calculado em relação ao protocolo **com split temporal**.

In [ ]:
def holdout_table(metric: str = "roc_auc") -> pd.DataFrame:
    rows = []
    ref = comparison["com_split_temporal"]["holdout"]
    for model in MODEL_ORDER:
        row = {"Modelo": MODEL_LABELS[model]}
        base = ref[model][metric]
        for key, label in PROTOCOLS.items():
            val = comparison[key]["holdout"][model][metric]
            row[label] = round(val, 4)
            if key != "com_split_temporal":
                row[f"Δ {label}"] = round(val - base, 4)
        rows.append(row)
    return pd.DataFrame(rows)


auc_df = holdout_table("roc_auc")
auc_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MODEL_ORDER))
width = 0.25
colors = ["#2ecc71", "#3498db", "#e74c3c"]

for i, (key, label) in enumerate(PROTOCOLS.items()):
    vals = [comparison[key]["holdout"][m]["roc_auc"] for m in MODEL_ORDER]
    ax.bar(x + i * width, vals, width, label=label, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
ax.set_ylim(0.85, 0.95)
ax.set_ylabel("ROC-AUC")
ax.set_title("Holdout ROC-AUC — comparação entre protocolos de split")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Holdout — F1-Score

| Modelo | Com split temporal | 80/20 cronológico | 80/20 aleatório |
|--------|------------------:|------------------:|----------------:|
| LightGBM | 0,668 | 0,692 | 0,685 |
| Random Forest | 0,669 | 0,698 | 0,703 |
| Árvore de Decisão | 0,673 | 0,689 | 0,697 |

In [ ]:
metrics = ["roc_auc", "f1", "precision", "recall", "accuracy"]
metric_labels = {
    "roc_auc": "ROC-AUC",
    "f1": "F1",
    "precision": "Precisão",
    "recall": "Recall",
    "accuracy": "Acurácia",
}

frames = []
for key, protocol in PROTOCOLS.items():
    for model in MODEL_ORDER:
        h = comparison[key]["holdout"][model]
        frames.append({
            "Protocolo": protocol,
            "Modelo": MODEL_LABELS[model],
            **{metric_labels[m]: round(h[m], 4) for m in metrics},
        })

full_df = pd.DataFrame(frames)
full_df.pivot(index="Modelo", columns="Protocolo", values="ROC-AUC")

## 4. Precisão vs Recall (trade-off)

| Modelo | Protocolo | Precisão | Recall |
|--------|-----------|--------:|-------:|
| LightGBM | Temporal | 0,523 | 0,926 |
| LightGBM | Aleatório | 0,563 | 0,877 |
| Random Forest | Temporal | 0,522 | 0,928 |
| Random Forest | Aleatório | 0,576 | 0,902 |

Com split temporal, os modelos de árvore priorizam **recall alto (~93%)** em detrimento de **precisão (~52%)** — metade dos alertas de subida são falsos positivos. O split aleatório melhora a precisão artificialmente ao misturar minutos passados e futuros no treino.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
styles = {
    "com_split_temporal": ("o", "#2ecc71", "Com split temporal"),
    "sem_split_aleatorio": ("s", "#e74c3c", "80/20 aleatório"),
}

for model in MODEL_ORDER:
    for key, (marker, color, label) in styles.items():
        h = comparison[key]["holdout"][model]
        ax.scatter(
            h["recall"], h["precision"],
            marker=marker, s=120, color=color,
            label=f"{MODEL_LABELS[model]} — {label}" if model == MODEL_ORDER[0] or key == "sem_split_aleatorio" else "",
        )
        ax.annotate(MODEL_LABELS[model][:3], (h["recall"], h["precision"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_title("Trade-off Precisão × Recall (temporal vs aleatório)")
ax.set_xlim(0.82, 0.95)
ax.set_ylim(0.50, 0.60)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc="lower left")
plt.tight_layout()
plt.show()

## 5. Validação cruzada temporal (TimeSeriesSplit)

Disponível apenas no protocolo **com split temporal** (5 folds no conjunto de treino).

| Modelo | ROC-AUC médio ± std |
|--------|---------------------|
| LightGBM | 0,892 ± 0,006 |
| Random Forest | 0,890 ± 0,005 |
| Árvore de Decisão | 0,875 ± 0,011 |

Para referência, a Regressão Logística com o mesmo protocolo (`04_modelos_regressao_logistica.ipynb`) obteve **0,887 ± 0,006** no TSCV e **0,894** no holdout.

In [ ]:
tscv_rows = []
for model in MODEL_ORDER:
    s = comparison["com_split_temporal"]["cv"][model]
    tscv_rows.append({
        "Modelo": MODEL_LABELS[model],
        "ROC-AUC (média)": round(s["mean"], 4),
        "ROC-AUC (std)": round(s["std"], 4),
    })
pd.DataFrame(tscv_rows)

## 6. Leitura principal

1. **Split aleatório infla AUC em ~1–2 p.p.** — o treino "vê" minutos futuros do mesmo mercado; as métricas ficam otimistas demais.
2. **Split temporal é mais conservador e honesto** — simula previsão em dias que o modelo nunca viu no holdout (10–11/03).
3. **80/20 cronológico** fica entre os dois: AUC um pouco abaixo do temporal, F1/precisão maiores porque o teste inclui mais dias (não só 10–11/03).
4. **Ranking estável:** Random Forest > LightGBM > Árvore de Decisão nos três protocolos.
5. **TSCV** confirma estabilidade: RF com menor desvio padrão (0,005); Árvore de Decisão é a menos estável (0,011).
6. **Baseline ingênua** (sempre classe 0): acurácia ~73,2%. Todos os protocolos superam isso, mas o split temporal é o único que avalia generalização temporal real.

**Conclusão:** para o trabalho final, reporte métricas do **split temporal**. Splits sem restrição temporal (especialmente aleatório) **superestimam** o desempenho real.

## 7. Expansão do dataset via API Polymarket

Tentamos ampliar a janela de dados usando `polymarket_client.py` e o pipeline `backend/pipelines/01_api_collect.py`.

### O que a API consegue fornecer

- Preço por minuto (`/prices-history`, fidelity=60)
- Volume e trades agregados (`/trades`)
- Catálogo de mercados movimentados (`discover_liquid_markets`)

### O que **não** está disponível retroativamente

A API pública **não expõe histórico de orderbook**. Faltam 6 colunas obrigatórias do parquet Kaggle:

- `mean_spread`, `close_spread`
- `bid_depth`, `ask_depth`, `depth_imbalance`
- `order_flow_imbalance`

`fetch_orderbook()` retorna apenas snapshot **atual**. Sem gravação contínua do livro de ofertas, não é possível reproduzir a microestrutura completa retroativamente.

### Resultado da coleta parcial

| Métrica | Dataset Kaggle | Coleta API |
|---------|----------------|------------|
| Janela | 06–11/03/2026 (6 dias) | ~27/05–27/06/2026 (~31 dias) |
| Mercados | 4.710 | 7 (de 20 solicitados) |
| Linhas | 5,5M | 5.068 |
| Schema completo | Sim (16 colunas) | Não (10 colunas parciais) |

**Conclusão:** não é possível aumentar a janela **mantendo a estrutura completa** só com a API. Para ampliar o dataset com as 12 features atuais, é necessário atualização do dataset Kaggle tick-level ou gravação contínua de orderbook daqui em diante.

In [ ]:
feas_path = Path("data/raw/api_collection/feasibility_report.json")
if feas_path.exists():
    feas = json.loads(feas_path.read_text(encoding="utf-8"))
    print("Schema completo reconstruível:", feas["full_schema_reconstructible"])
    print("Colunas ausentes na API:", ", ".join(feas["missing_columns"]))
    if feas.get("api_rows"):
        print(f"Linhas coletadas: {feas['api_rows']:,}")
        print(f"Período API: {feas.get('api_date_min')} -> {feas.get('api_date_max')}")
else:
    print("Execute: python backend/pipelines/01_api_collect.py --mode discover --max-markets 20")

## 8. Síntese para o trabalho final

| Aspecto | Recomendação |
|---------|-------------|
| Protocolo de avaliação | Holdout por data + `TimeSeriesSplit` |
| Métrica principal | ROC-AUC no holdout temporal |
| Modelo principal | Random Forest (AUC 0,911) ou LightGBM (AUC 0,906) |
| Evitar | `train_test_split(shuffle=True)` e KFold sem restrição temporal |
| Expansão de dados | Dataset Kaggle atualizado ou coleta contínua de orderbook |

Scripts relacionados:
- `scripts/compare_split_protocols.py` — comparação completa (inclui CV)
- `scripts/run_split_comparison_fast.py` — comparação rápida reutilizando cache
- `backend/pipelines/01_api_collect.py` — coleta parcial via API